# Comparación estadística final · F1

**experiment029.** Se recuperan los diez finalistas originales de experiment024, ordenados por ROC AUC, sin reentrenar. Se conserva el umbral cruzado con objetivo de recall del 98 % del informe original. Se comparan los 45 pares mediante bootstrap pareado y se corrigen sus p-valores con Holm. Los nombres son exactamente los del ranking original.

El F1 se calcula sobre todas las transacciones OOF, $F_1=2VP/(2VP+FP+FN)$. Se remuestrean positivos y negativos por separado, compartiendo los índices de cada réplica entre todos los modelos. Los modelos y umbrales permanecen fijos. El p bilateral aproximado se calcula centrando la distribución bootstrap de la diferencia en el nulo; los intervalos percentiles son puntuales del 95 %. La inferencia es exploratoria, condicionada a las OOF seleccionadas, no una prueba externa independiente.


In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import f1_score

SEMILLA = 42
B = 100_000
ALPHA = .05
OBJETIVO_RECALL = .98
RAIZ = Path.cwd()
if not (RAIZ/'credit_card_fraud.ipynb').exists() and (RAIZ/'final_project'/'credit_card_fraud.ipynb').exists():
    RAIZ = RAIZ/'final_project'
RAIZ = next((p for p in [RAIZ,*RAIZ.parents] if (p/'experiments/experiment026/resultados/top10.csv').is_file()),RAIZ)
if not (RAIZ/'experiments/experiment026/resultados/top10.csv').is_file():
    raise FileNotFoundError('Abrir Jupyter en la raíz de Credit-Fraud.')
CHECKPOINTS = RAIZ/'outputs/experiment024'
REFERENCIA = RAIZ/'experiments/experiment026/resultados'
SALIDA = RAIZ/'outputs/experiment029'
FIGURAS = SALIDA/'figuras'
FIGURAS.mkdir(parents=True,exist_ok=True)
sns.set_theme(style='whitegrid',palette='dark',context='notebook')
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':150,'axes.titlesize':12,'axes.titleweight':'semibold'})
COLORES = sns.color_palette('dark')

def cerrar_figura(fig,nombre):
    fig.tight_layout()
    fig.savefig(FIGURAS/f'{nombre}.png',bbox_inches='tight',dpi=150)
    fig.savefig(FIGURAS/f'{nombre}.pdf',bbox_inches='tight')
    plt.show();plt.close(fig)

sys.path.insert(0,str(RAIZ/'experiments/experiment027'))
sys.path.insert(0,str(RAIZ/'experiments/experiment029'))
from recall_inference import read_labels,read_oof,decisions,p_matrix
from f1_inference import pairwise_f1


## 1. Recuperar las predicciones originales y calcular F1

In [ ]:
ranking=pd.read_csv(REFERENCIA/'top10.csv')
manifest=json.loads((REFERENCIA/'resumen_final.json').read_text(encoding='utf-8'))
y=read_labels(RAIZ)
P,folds,audit=read_oof(CHECKPOINTS,ranking,manifest,y)
nombres=ranking.familia.tolist()
referencia=nombres[0]
Z,umbrales=decisions(y,P,folds,target=OBJETIVO_RECALL)
# La función histórica devuelve códigos; sustituirlos por los nombres originales.
umbrales['modelo']=np.tile(nombres,len(folds))
modelos,pares,patrones=pairwise_f1(y,Z,nombres,B=B,seed=SEMILLA,alpha=ALPHA)
np.testing.assert_allclose(modelos.f1,[f1_score(y,Z[:,j]) for j in range(len(nombres))],rtol=0,atol=1e-14)
assert len(pares)==45

# Verificar el F1 frente al informe original si está disponible localmente.
original=RAIZ/'outputs/informe_experiment024_es/metricas_alerta_cruzadas.csv'
if original.is_file():
    tabla=pd.read_csv(original).set_index('Código')
    np.testing.assert_allclose(modelos.f1,tabla.loc[nombres,'F1 cruzado'],rtol=0,atol=5e-10)
modelos=modelos.merge(ranking[['familia','puesto','roc_auc']],left_on='modelo',right_on='familia',validate='one_to_one').drop(columns='familia')
modelos.to_csv(SALIDA/'f1_modelos.csv',index=False)
pares.to_csv(SALIDA/'f1_pares.csv',index=False)
umbrales.to_csv(SALIDA/'umbrales.csv',index=False)
display(modelos[['puesto','modelo','f1']].style.format({'f1':'{:.8f}'}))
print('Referencia:',referencia)


## 2. Dashboard · p-valores originales y corrección de Holm

Cada celda representa un par de modelos. Se muestran los p-valores originales, sin binarización, con la misma escala lineal de 0 a 1. Los asteriscos indican p ≤ 0,05. Holm se aplica una sola vez a los 45 pares.

In [ ]:
fig,ejes=plt.subplots(1,2,figsize=(25,12))
mask=np.triu(np.ones((len(nombres),len(nombres)),dtype=bool))
# Mantener el nombre completo; solo separar familia y representación en dos líneas.
etiquetas=[n.replace(' | ',' |\n') for n in nombres]
for ax,col,titulo in zip(ejes,['p_bootstrap','p_holm'],['p-valores · bootstrap pareado','p-valores · corrección de Holm']):
    M=p_matrix(pares,nombres,col)
    M.to_csv(SALIDA/f'f1_{col}_matriz.csv')
    annotations=M.map(lambda p:'' if pd.isna(p) else f'{p:.4g}'+('*' if p<=ALPHA else ''))
    sns.heatmap(M,mask=mask,annot=annotations,fmt='',cmap='rocket',vmin=0,vmax=1,
        square=True,linewidths=.4,linecolor='white',cbar=False,ax=ax,
        xticklabels=etiquetas,yticklabels=etiquetas,annot_kws={'size':10})
    ax.set_title(titulo)
    ax.set(xlabel='',ylabel='')
    ax.tick_params(axis='x',labelrotation=70,labelsize=9)
    ax.tick_params(axis='y',labelrotation=0,labelsize=9)
    for tick in ax.get_xticklabels(): tick.set_ha('right')
fig.suptitle('Comparación pareada de F1 · diez finalistas',fontsize=15,fontweight='semibold')
fig.tight_layout(rect=(0,0,.95,.96))
cax=fig.add_axes([.96,.30,.012,.40])
fig.colorbar(ejes[0].collections[0],cax=cax,label='p-valor')
fig.text(.5,.015,'* p ≤ 0,05',ha='center')
fig.savefig(FIGURAS/'f1_dashboard_pvalues.png',bbox_inches='tight',dpi=150)
fig.savefig(FIGURAS/'f1_dashboard_pvalues.pdf',bbox_inches='tight')
plt.show();plt.close(fig)


## 3. Diferencia de F1 respecto al ganador por ROC AUC

Cada punto es F1 del modelo menos F1 de la referencia; las barras son intervalos bootstrap del 95 % (puntuales, no simultáneos). A la derecha de cero hay mejora; a la izquierda, deterioro. El asterisco indica significación después de Holm sobre los 45 pares.

In [ ]:
comparacion=pares.loc[pares.columna==referencia].set_index('fila').loc[nombres[1:]].reset_index()
comparacion.to_csv(SALIDA/'f1_vs_referencia.csv',index=False)
fig,ax=plt.subplots(figsize=(14,6.5))
pos=np.arange(len(comparacion))
for k,row in enumerate(comparacion.itertuples(index=False)):
    color=COLORES[0] if row.p_holm<=ALPHA else COLORES[7]
    ax.hlines(k,row.ci_low,row.ci_high,color=color,lw=2)
    ax.vlines([row.ci_low,row.ci_high],k-.075,k+.075,color=color,lw=1.5)
sns.scatterplot(data=comparacion.assign(pos=pos),x='delta_f1',y='pos',color=COLORES[0],s=60,ax=ax,zorder=3,legend=False)
ax.axvline(0,color='gray',ls='--',lw=1.2)
labels=[n+(' *' if p<=ALPHA else '') for n,p in zip(comparacion.fila,comparacion.p_holm)]
ax.set(yticks=pos,yticklabels=labels,xlabel='Diferencia de F1 (modelo − referencia)',ylabel='',
       title='Diferencia de F1 · IC bootstrap pareado del 95 %')
ax.invert_yaxis()
ax.set_title('Diferencia de F1 · IC bootstrap pareado del 95 %\nReferencia: '+referencia,pad=12)
cerrar_figura(fig,'f1_intervalos_vs_referencia')
meta=dict(experiment='experiment029',source='experiment024',source_signature=manifest['firma'],
          referencia=referencia,metric='F1',target_recall=OBJETIVO_RECALL,
          B=B,seed=SEMILLA,alpha=ALPHA,n_pares=len(pares),**patrones,
          method='stratified paired pattern bootstrap; centered two-sided p; Holm; pointwise percentile CI')
(SALIDA/'manifest.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print('Resultados:',SALIDA)
